In [ ]:
load_ext jupyter_black

In [ ]:
from copy import deepcopy
import numpy as np
import os
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import statsmodels.api as sm
from statsmodels.multivariate.multivariate_ols import _MultivariateOLS
import statsmodels.formula.api as smf
import matplotlib.patches as mpatches
from sklearn.linear_model import ElasticNetCV
from sklearn.preprocessing import StandardScaler
from sklearn.compose import ColumnTransformer
import pyfixest as pf

from dim_erasure import binarize_df

In [ ]:
def scale_data(df):
    # from https://github.com/CJTAYL/elastic_net_medium/blob/main/data_cleaning.py
    """
    Function to scale numerical data

    Parameters
    ----------
    df: dataframe
        Dataframe containing true labels of groups (clusters)

    Returns
    ----------
    df_scaled: dataframe
        Dataframe containing scaled values of numeric variables
    """
    numeric_columns = df.select_dtypes(include=["float64", "int"]).columns
    categorical_columns = df.select_dtypes(exclude=["float64", "int"]).columns
    ct = ColumnTransformer(
        [("scale", StandardScaler(), numeric_columns)], remainder="passthrough"
    )

    # Fit and transform the data
    df_scaled_array = ct.fit_transform(df)

    # ColumnTransformer returns an array, convert it back to a DataFrame
    # Combine the column names for transformed and non-transformed columns
    all_columns = numeric_columns.tolist() + categorical_columns.tolist()
    df_scaled = pd.DataFrame(df_scaled_array, columns=all_columns, index=df.index)

    return df_scaled

In [ ]:
demographics = {
    "prism": [
        "age",
        "gender",
        "employment_status",
        "education",
        "marital_status",
        "english_proficiency",
        "religion",
        "ethnicity",
        "birth_region",
        "reside_region",
        "lm_familiarity",
    ],
    "chen": [
        "Gender",
        "human_Gender",
    ],
    "cad_en": [
        "annotator_age",
        "annotator_gender",
        "annotator_education_level",
        "annotator_political",
        "annotator_ethnicity",
    ],
    "cad_fr": [
        "annotator_age",
        "annotator_gender",
        "annotator_education_level",
        "annotator_political",
        "annotator_ethnicity",
    ],
    "cad_pt": [
        "annotator_age",
        "annotator_gender",
        "annotator_education_level",
        "annotator_political",
        "annotator_ethnicity",
    ],
    "cad_it": [
        "annotator_age",
        "annotator_gender",
        "annotator_education_level",
        "annotator_political",
        "annotator_ethnicity",
    ],
}

domains = ["legal", "salary", "medical", "benefits", "political"]

In [ ]:
questions = pd.read_pickle("data/Llama-3.1-8B-Instruct_questions.gz")
questions_correct_answers = dict(zip(questions.q_id, questions.correct_answer))
domain_qid_map = {
    domain: questions.loc[questions["domain"] == domain, "q_id"].tolist()
    for domain in domains
}

In [ ]:
for dataset in [
    "cad_en",
    "prism",
]:
    all_cols = deepcopy(demographics[dataset])
    df = pd.read_pickle(f"llama_beliefs/Llama-3.1-8B-Instruct_{dataset}_answers.gz")
    df = df.rename(columns={"label": "Gender"})

    for c in [qid for d in domains for qid in domain_qid_map[d] if d != "salary"]:
        df[c] = 1 * (df[c].str.lower() == questions_correct_answers[c])

    for c in domain_qid_map["salary"]:
        df[c] = df[c].str.replace(",", "").str.extract(r"^[^\d]*(\d+)").astype(float)

    for domain in domains:
        df[domain] = df[[qid for qid in domain_qid_map[domain]]].mean(axis=1)
        if domain != "salary":
            df[domain] = df[domain] * 100

    df = df.drop(
        columns=[f"q_{i}" for i in range(50)]
        + ["q_59", "q_60"]
        + [f"q_{i}" for i in range(61, 211)]
    )

    df_linguistic = pd.read_pickle(
        f"data/{dataset + '_utterances' if dataset != 'chen' else dataset}_linguistic.gz"
    ).drop(
        columns=[
            "s_neutral_model_response",
            "s_neutral_user_prompt",
            "model_response_liwc_Segment",
            "user_prompt_liwc_Segment",
        ],
        errors="ignore",
    )
    for c in ["politeness_user_prompt", "politeness_model_response"]:
        if c in df_linguistic:
            df_linguistic[c] = df_linguistic[c].replace(
                {"impolite": 0, "neutral": 0.5, "polite": 1, "somewhat polite": 0.75}
            )
    df_linguistic = df_linguistic.rename(columns={"gpt_description": "topic"})
    all_cols += ["topic"]
    group_cols = ["conversation_id"] + all_cols
    df_linguistic = (
        df_linguistic.groupby(group_cols)[
            [
                c
                for c in df_linguistic.columns
                if ("model_response" in c or "user_prompt" in c)
                and (c not in ["model_response", "user_prompt"])
            ]
        ]
        .mean()
        .reset_index()
    )
    df = df.merge(
        df_linguistic[
            ["conversation_id"]
            + [
                c
                for c in df_linguistic.columns
                if "model_response" in c or "user_prompt" in c or c == "topic"
            ]
        ],
        on="conversation_id",
    )

    scaled_df = scale_data(df)
    scaled_df[domains] = df[domains]
    scaled_df = scaled_df.loc[scaled_df.duplicated(subset=["topic"], keep=False)]
    scaled_df = pd.get_dummies(
        scaled_df, columns=all_cols, prefix=["dummy_" + c for c in all_cols]
    )
    all_cols += [
        c for c in scaled_df.columns if "model_response" in c or "user_prompt" in c
    ]

    for col in domains:
        filtered_df = scaled_df.loc[~scaled_df[col].isna()]
        filtered_df = filtered_df[
            [c for c in filtered_df.columns if c in all_cols or c.startswith("dummy_")]
            + [col]
        ].dropna()

        l1_ratios = [0.1, 0.5, 0.7, 0.9, 0.95, 0.99, 1]  # α candidates
        model = ElasticNetCV(
            l1_ratio=l1_ratios,
            cv=5,
            max_iter=10_000,
            random_state=42,
        )
        feature_names = [
            c for c in filtered_df.columns if c in all_cols or c.startswith("dummy_")
        ]

        best_model = model.fit(
            filtered_df[
                [
                    c
                    for c in filtered_df.columns
                    if c in all_cols or c.startswith("dummy_")
                ]
            ],
            filtered_df[col],
        )

        best_alpha = model.alpha_
        best_l1 = model.l1_ratio_
        coefs = model.coef_
        score = best_model.score(
            filtered_df[
                [
                    c
                    for c in filtered_df.columns
                    if c in all_cols or c.startswith("dummy_")
                ]
            ],
            filtered_df[col],
        )

        print(f"Best α (alpha) : {best_alpha:.4f}")
        print(
            f"Best λ (l1_ratio): {best_l1:.2f}  "
            f"({'pure Lasso' if best_l1==1 else 'pure Ridge' if best_l1==0 else 'Elastic Net'})"
        )
        print(f"Non-zero coefficients: {np.sum(coefs != 0)} / {len(coefs)}")
        print(f"Score: {score}")

        # ── 4. Filter to non-zero (selected) features ────────────────────────────────
        mask = coefs != 0
        sel_names = np.array(feature_names)[mask]
        sel_coefs = coefs[mask]

        # Sort by absolute value descending
        order = np.argsort(np.abs(sel_coefs))[::-1]
        sel_names = sel_names[order]
        sel_coefs = sel_coefs[order]

        sel_names = [s.split("dummy_")[-1][:50] for s in sel_names]

        # ── 5. Plot — 8-column layout ─────────────────────────────────────────────────
        POS_COLOR = "#2E86AB"  # blue  – positive coefficients
        NEG_COLOR = "#E84855"  # red   – negative coefficients
        TEXT_COLOR = "#0F1117"
        GRID_COLOR = "#b4faba"
        BG_COLOR = "#E8EAF0"

        N_COLS = 8
        n_feats = len(sel_coefs)
        n_rows = int(np.ceil(n_feats / N_COLS))

        # Split into columns (most important → left column)
        col_names = [sel_names[i * n_rows : (i + 1) * n_rows] for i in range(N_COLS)]
        col_coefs = [sel_coefs[i * n_rows : (i + 1) * n_rows] for i in range(N_COLS)]

        fig, axes = plt.subplots(
            1,
            N_COLS,
            figsize=(42 if dataset == "prism" else 72, max(4, n_rows * 0.55 + 1.5)),
        )
        fig.patch.set_facecolor(BG_COLOR)

        for col_idx, (ax, names, coefs_col) in enumerate(
            zip(axes, col_names, col_coefs)
        ):
            ax.set_facecolor(BG_COLOR)
            if len(names) == 0:
                ax.set_visible(False)
                continue

            y_pos = np.arange(len(names))
            colors = [POS_COLOR if c > 0 else NEG_COLOR for c in coefs_col]

            bars = ax.barh(
                y_pos, coefs_col, color=colors, height=0.65, edgecolor="none", zorder=3
            )

            # if col_idx != 0:
            #     ax.set_xlim(axes[0].get_xlim())

            # Value labels
            x_range = max(abs(coefs_col)) if len(coefs_col) else 1
            for bar, val in zip(bars, coefs_col):
                ax.text(
                    val + 0.04 * x_range * np.sign(val),
                    bar.get_y() + bar.get_height() / 2,
                    f"{val:+.3f}",
                    va="center",
                    ha="left" if val > 0 else "right",
                    fontsize=8,
                    color=TEXT_COLOR,
                    alpha=0.85,
                )

            ax.set_yticks(y_pos)
            ax.set_yticklabels(names, fontsize=9.5, color=TEXT_COLOR)
            ax.tick_params(axis="x", colors=TEXT_COLOR, labelsize=8)
            ax.axvline(0, color=TEXT_COLOR, linewidth=0.8, alpha=0.4, zorder=2)
            ax.grid(axis="x", color=GRID_COLOR, linewidth=0.6, zorder=1)
            ax.spines[["top", "right", "left", "bottom"]].set_visible(False)
            ax.invert_yaxis()

            # Column rank label
            start_rank = col_idx * n_rows + 1
            end_rank = min(start_rank + len(names) - 1, n_feats)
            rank_label = (
                f"#{start_rank}"
                if start_rank == end_rank
                else f"#{start_rank}–#{end_rank}"
            )
            ax.set_title(rank_label, color=TEXT_COLOR, fontsize=9, alpha=0.5, pad=6)

            if col_idx == 0:
                ax.set_xlabel("Coefficient", color=TEXT_COLOR, fontsize=9, labelpad=6)

        # Title & subtitle
        fig.text(
            0.5,
            0.95,
            "Elastic Net — Selected Features & Coefficients",
            fontsize=14,
            fontweight="bold",
            color=TEXT_COLOR,
            va="top",
            ha="center",
        )
        fig.text(
            0.5,
            0.935,
            f"α = {best_alpha:.4f}   |  λ  (l1_ratio) = {best_l1:.2f}   |   score = {score:.2f} "
            f"{n_feats} of {len(model.coef_)} features selected   |   5-fold CV",
            fontsize=8.5,
            color=TEXT_COLOR,
            alpha=0.6,
            va="top",
            ha="center",
        )

        # Legend
        pos_patch = mpatches.Patch(color=POS_COLOR, label="Positive effect")
        neg_patch = mpatches.Patch(color=NEG_COLOR, label="Negative effect")
        fig.legend(
            handles=[pos_patch, neg_patch],
            loc="lower center",
            ncol=2,
            frameon=False,
            fontsize=9,
            labelcolor=TEXT_COLOR,
            bbox_to_anchor=(0.5, 0.01),
        )

        plt.tight_layout(rect=[0, 0.06, 1, 0.92])

        plt.savefig(
            f"elastic_net_figures/{dataset}_{col}.png",
            dpi=150,
            bbox_inches="tight",
            facecolor=BG_COLOR,
        )
        plt.show()
        print(f"Plot saved to elastic_net_figures/{dataset}_{col}.png")